In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
pyspark.__version__

'3.5.3'

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 16:44:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-11 16:44:42--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.98, 13.35.33.60, 13.35.33.83, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.98|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.2’

yellow_tripdata_202 100%[===================>]  67.84M  1.96MB/s    in 33s     

2026-03-11 16:45:15 (2.04 MB/s) - ‘yellow_tripdata_2025-11.parquet.2’ saved [71134255/71134255]



In [6]:
df_yellow = spark.read.parquet('file:///home/hadoop/06-batch/yellow_tripdata_2025-11.parquet')

In [7]:
df_yellow.count()

4181444

In [ ]:
df_yellow.repartition(4).write.parquet('file:///home/hadoop/06-batch/data/raw/yellow/2025/11/', mode='overwrite')

In [ ]:
!ls -lh data/raw/yellow/2025/11/

total 98M
-rw-r--r-- 1 hadoop hadoop 25M Mar 11 16:24 part-00000-a8693c15-f812-4152-b0cc-3f8ec611eab4-c000.snappy.parquet
-rw-r--r-- 1 hadoop hadoop 25M Mar 11 16:24 part-00001-a8693c15-f812-4152-b0cc-3f8ec611eab4-c000.snappy.parquet
-rw-r--r-- 1 hadoop hadoop 25M Mar 11 16:24 part-00002-a8693c15-f812-4152-b0cc-3f8ec611eab4-c000.snappy.parquet
-rw-r--r-- 1 hadoop hadoop 25M Mar 11 16:24 part-00003-a8693c15-f812-4152-b0cc-3f8ec611eab4-c000.snappy.parquet
-rw-r--r-- 1 hadoop hadoop   0 Mar 11 16:24 _SUCCESS


In [ ]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [20]:
# What is the length of the longest trip in the dataset in hours?
from pyspark.sql import functions as F

df_yellow \
    .withColumn(
        'trip_length_hours',
        (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / F.lit(3600.0)
    ) \
    .agg(F.max('trip_length_hours').alias('max_trip_length_hours')) \
    .show()

+---------------------+
|max_trip_length_hours|
+---------------------+
|    90.64666666666666|
+---------------------+



In [ ]:
from pyspark.sql import functions as F

# Filter for November 15, 2025
df_nov_15 = df_yellow.filter(F.to_date(F.col('tpep_pickup_datetime')) == '2025-11-15')

df_nov_15.count()

162604

In [8]:
df_zones = spark.read.csv('file:///home/hadoop/06-batch/taxi_zone_lookup.csv', header=True, inferSchema=True)

In [9]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [14]:
df_yellow.registerTempTable('yellow')
df_zones.registerTempTable('zones')

/home/hadoop/06-batch/.venv/lib/python3.12/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [16]:
# Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?
spark.sql("""
SELECT z.Zone, COUNT(*) AS pickup_count
FROM yellow y
JOIN zones z ON y.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY pickup_count ASC
""").show()

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
|Eltingville/Annad...|           1|
|       Arden Heights|           1|
|       Port Richmond|           3|
|       Rikers Island|           4|
|   Rossville/Woodrow|           4|
| Green-Wood Cemetery|           4|
|         Great Kills|           4|
|         Jamaica Bay|           5|
|         Westerleigh|          12|
|        Crotona Park|          14|
|             Oakwood|          14|
|New Dorp/Midland ...|          14|
|       West Brighton|          14|
|       Willets Point|          15|
|Breezy Point/Fort...|          16|
|Saint George/New ...|          17|
|       Broad Channel|          18|
|     Mariners Harbor|          21|
|Heartland Village...|          22|
+--------------------+------------+
only showing top 20 rows

